# NVIDIA Studio Voice - Python Notebook

This notebook demonstrates how to use the NVIDIA Studio Voice service with Python. NVIDIA Studio Voice (SV) is a speech enhancement model from NVIDIA. Studio Voice enhances input speech recorded with low-quality microphones in noisy or reverberant environments, producing studio-quality speech in real-time.

## Overview

The service processes audio files (WAV format) to output enhanced audio with improved clarity and reduced noise:

- **Service Integration**: Connect to Studio Voice services
- **Audio Enhancement**: AI-powered speech enhancement
- **Multiple Models**: Support for high-quality (HQ) and low-latency (LL) models
- **Streaming Support**: Process audio in both transactional and streaming modes
- **Flexible Sample Rates**: Support for 16kHz and 48kHz audio

## Requirements

- **Input Audio**: WAV files at 16kHz or 48kHz sample rate
- **Output**: Enhanced WAV audio files
- **Service**: Access to a running Studio Voice service instance
- **Model Types**: 
  - `48k-hq`: 48kHz High Quality model (6-second chunks)
  - `48k-ll`: 48kHz Low Latency model (10ms chunks)
  - `16k-hq`: 16kHz High Quality model (6-second chunks)


## Installation

### Requirements:
- Python 3.10 or above
- pip package manager
- gRPC dependencies from the requirements.txt file


In [1]:
!pip install -r ../requirements.txt

Defaulting to user installation because normal site-packages is not writeable


## Service Configuration

Configure the connection to your Studio Voice service. The service can be running on your machine or on a remote server accessible from your environment.
To run the Studio Voice NIM follow the instructions in [Getting Started](https://docs.nvidia.com/nim/maxine/studio-voice/latest/index.html).


In [2]:
import os
import sys
import pathlib

# Setup paths for Studio Voice modules
SCRIPT_PATH = str(pathlib.Path().resolve())
print(SCRIPT_PATH)
sys.path.append(os.path.join(SCRIPT_PATH, "../scripts"))
sys.path.append(os.path.join(SCRIPT_PATH, "../interfaces/studio_voice"))
sys.path.append(os.path.join(SCRIPT_PATH, "../../"))

# Service connection configuration
SERVICE_HOST = "localhost"  # Update to your service host
SERVICE_PORT = 8001         # Update to your service port
SERVICE_TARGET = f"{SERVICE_HOST}:{SERVICE_PORT}"

print(f"Service target configured: {SERVICE_TARGET}")
print(f"Python paths configured for Studio Voice modules")


/localhome/local-nimenon/maxine-client-packages/studio-voice/notebook
Service target configured: localhost:8001
Python paths configured for Studio Voice modules


## Import Libraries

Import the required libraries for gRPC communication, audio processing, and the Studio Voice protobuf modules.

In [3]:
import grpc
import time
import soundfile as sf
import numpy as np
from typing import Iterator
from tqdm.auto import tqdm
from IPython.display import Audio, display

# Import Studio Voice modules
import studiovoice_pb2
import studiovoice_pb2_grpc

print("All libraries imported successfully!")


All libraries imported successfully!


## Helper Functions

Define functions for processing audio data and communicating with the Studio Voice service. These functions handle the gRPC streaming protocol used by the Studio Voice service.

### Function Overview

The Studio Voice service uses gRPC streaming for audio enhancement. The implementation consists of two main functions:

1. **Request Generator**: A Python iterator that yields request chunks to stream to the service
2. **Response Processor**: A function that processes the incoming gRPC data stream and writes the output file

### Streaming Protocol Details

- **Streaming Mode**: Audio is split into chunks (10ms for low-latency models, 6 seconds for high-quality models)
- **Transactional Mode**: Entire audio file is sent in 64KB chunks
- **Response Stream**: Enhanced audio data chunks containing the processed output WAV file


In [4]:
def generate_request_for_inference(
    input_filepath: str,
    model_type: str,
    sample_rate: int,
    streaming: bool
) -> Iterator[studiovoice_pb2.EnhanceAudioRequest]:
    """Generate streaming requests for the Studio Voice service.
    
    This function creates a Python iterator that yields gRPC request messages for streaming
    to the Studio Voice service. The chunking strategy depends on the mode and model type.
    
    Args:
        input_filepath: Path to the input WAV audio file
        model_type: Model type to use (48k-hq, 48k-ll, or 16k-hq)
        sample_rate: Audio sample rate (16000 or 48000)
        streaming: If True, use streaming mode; if False, use transactional mode
        
    Yields:
        EnhanceAudioRequest messages containing audio data chunks
        
    Raises:
        FileNotFoundError: If input file doesn't exist
        ValueError: If sample rate doesn't match the input file
    """
    
    # Validate input file exists
    if not os.path.exists(input_filepath):
        raise FileNotFoundError(f"Audio file not found: {input_filepath}")
    
    if streaming:
        # Streaming mode: chunk audio based on model type
        # High quality models require 6sec input
        # Low latency models require 10ms input chunk
        input_audio, sample_rate_file = sf.read(input_filepath)
        input_audio = input_audio.astype(np.float32)  # Convert to float32
        
        # Validate sample rate
        if sample_rate_file != sample_rate:
            raise ValueError(
                f"Sample rate mismatch: expected {sample_rate}, got {sample_rate_file}"
            )
        
        input_size_in_ms = 10 if (model_type == "48k-ll") else 6000
        samples_per_ms = sample_rate // 1000
        input_float_size = int(input_size_in_ms * samples_per_ms)
        
        # Pad audio to ensure it's divisible by chunk size
        pad_length = (input_float_size - len(input_audio) % input_float_size) % input_float_size
        if pad_length > 0:
            input_audio = np.pad(input_audio, (0, pad_length), "constant")
        
        print(f"Streaming mode: Chunk size = {input_size_in_ms}ms ({input_float_size} samples)")
        print(f"Total audio length: {len(input_audio)} samples")
        print(f"Number of chunks: {len(input_audio) // input_float_size}")
        
        # Stream audio in chunks
        for i in range(0, len(input_audio), input_float_size):
            data = input_audio[i : i + input_float_size]
            yield studiovoice_pb2.EnhanceAudioRequest(audio_stream_data=data.tobytes())
    else:
        # Transactional mode: send entire file in 64KB chunks
        DATA_CHUNKS = 64 * 1024  # bytes
        file_size = os.path.getsize(input_filepath)
        
        print(f"Transactional mode: Sending audio in {DATA_CHUNKS} byte chunks")
        print(f"Total file size: {file_size / 1024:.1f} KB")
        
        with open(input_filepath, "rb") as fd:
            while True:
                buffer = fd.read(DATA_CHUNKS)
                if buffer == b"":
                    break
                yield studiovoice_pb2.EnhanceAudioRequest(audio_stream_data=buffer)


def write_output_file_from_response(
    response_iter: Iterator[studiovoice_pb2.EnhanceAudioResponse],
    output_filepath: str,
    sample_rate: int,
    streaming: bool,
) -> int:
    """Write the output file from the incoming gRPC data stream.
    
    Args:
        response_iter: Responses from the server to write into output file
        output_filepath: Path to output file
        sample_rate: Audio sample rate for output file
        streaming: If True, collect float32 audio data; if False, write raw WAV data
        
    Returns:
        Number of responses processed (only in streaming mode)
    """
    print(f"Writing output to {output_filepath}")
    
    if streaming:
        # Streaming mode: collect audio chunks and write as WAV
        output_audio = []
        response_count = 0
        
        with tqdm(desc="Receiving audio chunks", unit="chunks", leave=False) as pbar:
            for response in response_iter:
                response_count += 1
                output_audio.append(np.frombuffer(response.audio_stream_data, np.float32))
                pbar.update(1)
        
        # Combine all chunks and write to file
        combined_audio = np.hstack(output_audio)
        sf.write(output_filepath, combined_audio, sample_rate)
        
        print(f"Completed: Received {response_count} chunks")
        print(f"Output duration: {len(combined_audio) / sample_rate:.2f} seconds")
        
        return response_count
    else:
        # Transactional mode: write raw WAV data
        total_bytes = 0
        
        with open(output_filepath, "wb") as fd:
            with tqdm(desc="Receiving audio data", unit="B", unit_scale=True, leave=False) as pbar:
                for response in response_iter:
                    if response.HasField("audio_stream_data"):
                        chunk_data = response.audio_stream_data
                        fd.write(chunk_data)
                        total_bytes += len(chunk_data)
                        pbar.update(len(chunk_data))
        
        print(f"Completed: Received {total_bytes / 1024:.1f} KB total")
        
        return 0


## Audio Processing

Process audio files with the Studio Voice service. This method runs the audio enhancement effect.


In [5]:
def process_studio_voice_audio(
    input_path: str,
    output_path: str,
    model_type: str = "48k-hq",
    streaming: bool = False
) -> bool:
    """Process audio with Studio Voice service.
    
    Args:
        input_path: Path to input audio file (WAV format)
        output_path: Path for output audio file
        model_type: Model type to use (48k-hq, 48k-ll, or 16k-hq)
        streaming: If True, use streaming mode; if False, use transactional mode
    
    Returns:
        True if processing succeeded, False otherwise
    """
    try:
        # Determine sample rate based on model type
        sample_rate = 16000 if model_type == "16k-hq" else 48000
        
        print(f"\nProcessing audio: {input_path}")
        print(f"Model type: {model_type}")
        print(f"Sample rate: {sample_rate} Hz")
        print(f"Mode: {'Streaming' if streaming else 'Transactional'}")
        print(f"Connecting to service: {SERVICE_TARGET}")
        
        # Validate input file exists
        if not os.path.isfile(input_path):
            raise FileNotFoundError(f"Input file not found: {input_path}")
        
        # Check the sample rate of the input audio file
        input_info = sf.info(input_path)
        input_sample_rate = input_info.samplerate
        print(f"Input file sample rate: {input_sample_rate} Hz")
        
        # Validate sample rate matches
        if input_sample_rate != sample_rate:
            raise ValueError(
                f"Sample rate mismatch: model expects {sample_rate} Hz, "
                f"but input file has {input_sample_rate} Hz"
            )
        
        # Connect to service (use secure channel if needed)
        with grpc.insecure_channel(SERVICE_TARGET) as channel:
            stub = studiovoice_pb2_grpc.StudioVoiceStub(channel)
            
            start_time = time.time()
            
            # Process audio
            responses = stub.EnhanceAudio(
                generate_request_for_inference(
                    input_path, model_type, sample_rate, streaming
                )
            )
            
            # Write output with progress tracking
            response_count = write_output_file_from_response(
                responses, output_path, sample_rate, streaming
            )
            
            end_time = time.time()
            processing_time = end_time - start_time
            
            if streaming and response_count > 0:
                avg_latency = processing_time / response_count
                print(f"Average latency per chunk: {avg_latency*1000:.2f}ms")
            
            print(f"Processing complete in {processing_time:.2f}s")
            print(f"Output saved: {output_path}")
            
            return True
            
    except FileNotFoundError as e:
        print(f"Input file not found: {e}")
        print("   Please update file path with a valid audio file")
        return False
    except ValueError as e:
        print(f"Configuration error: {e}")
        return False
    except grpc.RpcError as e:
        print(f"Service connection failed: {e}")
        print(f"   Ensure the Studio Voice service is running at {SERVICE_TARGET}")
        return False
    except Exception as e:
        print(f"Processing failed: {e}")
        import traceback
        traceback.print_exc()
        return False


## Run Studio Voice Processing - Basic Example

This cell demonstrates how to run Studio Voice processing on an example audio file (.wav) using the transactional mode with the 48kHz high-quality model.

**Important Note:** Ensure that your Studio Voice NIM server is running with the same model type as specified in the processing command below. The model type hosted on the server must match the `model_type` parameter (`48k-hq`, `48k-ll`, or `16k-hq`). Mismatches between client and server model types will result in processing errors.


In [7]:
# Configuration
input_filepath = "../assets/studio_voice_48k_input.wav"    # Update with your input audio path
output_filepath = "studio_voice_48k_output.wav"             # Desired output audio path

# Process audio with default settings (48k-hq, transactional mode)
success = process_studio_voice_audio(
    input_path=input_filepath,
    output_path=output_filepath,
    model_type="48k-hq",
    streaming=False
)



Processing audio: ../assets/studio_voice_48k_input.wav
Model type: 48k-hq
Sample rate: 48000 Hz
Mode: Transactional
Connecting to service: localhost:8001
Input file sample rate: 48000 Hz
Transactional mode: Sending audio in 65536 byte chunks
Total file size: 1063.2 KB
Writing output to studio_voice_48k_output.wav


Receiving audio data: 0.00B [00:00, ?B/s]

Completed: Received 1063.2 KB total
Processing complete in 11.51s
Output saved: studio_voice_48k_output.wav


In [8]:
# Playback: compare input and output audio
if success:
    print("Input audio:")
    display(Audio(filename=input_filepath))
    print("\nEnhanced output audio:")
    display(Audio(filename=output_filepath))

Input audio:



Enhanced output audio:


## Advanced Usage Examples

Examples demonstrating different model types and processing modes.

**Important:** When running these examples, ensure that your Studio Voice NIM server is configured with the matching model type. The model type specified in the processing command (`48k-hq`, `48k-ll`, or `16k-hq`) must match the model type that the NIM server was started with.


### Model Types

Studio Voice supports three model types, each optimized for different use cases:

#### 48k-hq: 48kHz High Quality
- **Sample Rate**: 48000 Hz
- **Chunk Size**: 6 seconds (for streaming mode)
- **Use Case**: Best audio quality for professional applications
- **Processing**: Transactional or streaming mode

#### 48k-ll: 48kHz Low Latency
- **Sample Rate**: 48000 Hz
- **Chunk Size**: 10ms (for streaming mode)
- **Use Case**: Real-time applications requiring minimal latency
- **Processing**: Primarily used with streaming mode

#### 16k-hq: 16kHz High Quality
- **Sample Rate**: 16000 Hz
- **Chunk Size**: 6 seconds (for streaming mode)
- **Use Case**: Telephony, voice commands, bandwidth-constrained scenarios
- **Processing**: Transactional or streaming mode

### Processing Modes

#### Transactional Mode (Default)
- Entire audio file is sent at once in 64KB chunks
- Simpler implementation
- Suitable for batch processing and non-realtime applications

#### Streaming Mode
- Audio is processed in smaller chunks based on model type
- Enables real-time or near-realtime processing
- Provides per-chunk latency metrics
- Recommended for low-latency applications


### Example 1: Transactional Mode with 48k-hq Model

Standard high-quality processing for batch audio enhancement.


In [ ]:
# Example 1: Transactional Mode with 48k-hq Model
success = process_studio_voice_audio(
    input_path="../assets/studio_voice_48k_input.wav",
    output_path="output_48k_hq_transactional.wav",
    model_type="48k-hq",
    streaming=False
)


### Example 2: Streaming Mode with 48k-hq Model

High-quality processing with streaming for better progress tracking.


In [ ]:
# Example 2: Streaming Mode with 48k-hq Model
success = process_studio_voice_audio(
    input_path="../assets/studio_voice_48k_input.wav",
    output_path="output_48k_hq_streaming.wav",
    model_type="48k-hq",
    streaming=True
)


### Example 3: Low Latency Model (48k-ll) with Streaming

Low latency processing with 10ms chunks - ideal for real-time applications.

**Note:** Ensure your Studio Voice NIM is configured to use the `48k-ll` (low latency) model type. The server must be started with the low-latency model for this example to work correctly.


In [ ]:
# Example 3: Low Latency Model (48k-ll) with Streaming
success = process_studio_voice_audio(
    input_path="../assets/studio_voice_48k_input.wav",
    output_path="output_48k_ll_streaming.wav",
    model_type="48k-ll",
    streaming=True  # Low-latency model is typically used with streaming
)


### Example 4: 16kHz Model (16k-hq)

For telephony and voice command applications with 16kHz audio.

**Note:** This example requires:
1. A 16kHz input audio file - make sure your input file matches the sample rate
2. Your Studio Voice NIM must be configured to use the `16k-hq` model type - the server must be started with the 16kHz model


In [ ]:
# Example 4: 16kHz Model (16k-hq)
# Check if 16kHz input file exists, otherwise skip
input_16k = "../assets/studio_voice_16k_input.wav"
if os.path.exists(input_16k):
    success = process_studio_voice_audio(
        input_path=input_16k,
        output_path="output_16k_hq.wav",
        model_type="16k-hq",
        streaming=False
    )
else:
    print(f"Skipping: 16kHz input file not found at {input_16k}")
    print("  To test this model, provide a 16kHz WAV file and update the path.")


For more information on Studio Voice features, model configurations, and advanced usage, please refer to the [NVIDIA Studio Voice Documentation](https://docs.nvidia.com/nim/maxine/studio-voice/latest/index.html).